# Model Comparison & Benchmark Report

This notebook compares the performance of different anomaly detection models (Automata, LSTM, GRU, CNN1D) across datasets (SKAB, BATADAL) and scenarios (original, gaussian_noise, unseen).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Configure visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Load results
RESULTS_SKAB = Path('../results/skab')
RESULTS_BATADAL = Path('../results/batadal')

In [ ]:
# Load SKAB results
skab_summary_csv = RESULTS_SKAB / 'skab_results_summary.csv'
skab_results_json = RESULTS_SKAB / 'skab_results.json'

if skab_summary_csv.exists():
    df_skab = pd.read_csv(skab_summary_csv)
    print("📊 SKAB Results Summary:")
    print(df_skab.to_string(index=False))
else:
    print("❌ SKAB results not found. Run experiments first.")
    df_skab = None

In [ ]:
# Load BATADAL results
batadal_summary_csv = RESULTS_BATADAL / 'batadal_results_summary.csv'
batadal_results_json = RESULTS_BATADAL / 'batadal_results.json'

if batadal_summary_csv.exists():
    df_batadal = pd.read_csv(batadal_summary_csv)
    print("📊 BATADAL Results Summary:")
    print(df_batadal.to_string(index=False))
else:
    print("❌ BATADAL results not found. Run experiments first.")
    df_batadal = None

In [ ]:
# Create comparison matrices by scenario
if df_skab is not None:
    metrics = df_skab['metric'].unique()
    scenarios = df_skab['scenario'].unique()
    
    for metric in metrics:
        print(f"\n{'='*60}")
        print(f"SKAB - {metric.upper()} Comparison")
        print(f"{'='*60}")
        
        metric_df = df_skab[df_skab['metric'] == metric]
        matrix = metric_df.pivot_table(index='scenario', columns='model', values='mean')
        print(matrix.to_string())
        print()

In [ ]:
# Plot F1 scores by scenario (SKAB)
if df_skab is not None:
    f1_data = df_skab[df_skab['metric'] == 'f1']
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    scenarios = f1_data['scenario'].unique()
    x = np.arange(len(scenarios))
    width = 0.2
    
    models = f1_data['model'].unique()
    for i, model in enumerate(models):
        model_data = f1_data[f1_data['model'] == model]
        values = [model_data[model_data['scenario'] == s]['mean'].values[0] if len(model_data[model_data['scenario'] == s]) > 0 else 0 for s in scenarios]
        ax.bar(x + i*width, values, width, label=model)
    
    ax.set_xlabel('Scenario', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('SKAB: F1 Score Comparison by Scenario', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(scenarios)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_SKAB / 'skab_f1_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Figure saved to results/skab/")

In [ ]:
# Create subplots for all metrics (SKAB)
if df_skab is not None:
    metrics = df_skab['metric'].unique()
    scenarios = df_skab['scenario'].unique()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('SKAB: Model Performance Across All Metrics', fontsize=16, fontweight='bold')
    
    axes = axes.flatten()
    
    for ax_idx, metric in enumerate(sorted(metrics)):
        ax = axes[ax_idx]
        metric_df = df_skab[df_skab['metric'] == metric]
        
        for model in metric_df['model'].unique():
            model_data = metric_df[metric_df['model'] == model]
            model_data = model_data.sort_values('scenario')
            ax.plot(model_data['scenario'], model_data['mean'], marker='o', label=model, linewidth=2)
        
        ax.set_xlabel('Scenario')
        ax.set_ylabel('Score')
        ax.set_title(f'{metric.upper()} Score')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(RESULTS_SKAB / 'skab_all_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Comprehensive metrics figure saved")

In [ ]:
# Heatmap: Model performance by scenario and metric (SKAB)
if df_skab is not None:
    # Focus on one model at a time for clarity
    models = df_skab['model'].unique()
    
    for model in sorted(models):
        model_df = df_skab[df_skab['model'] == model]
        heatmap_data = model_df.pivot_table(index='scenario', columns='metric', values='mean')
        
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Score'})
        ax.set_title(f'SKAB: {model.upper()} Performance Heatmap', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(RESULTS_SKAB / f'skab_{model}_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    print("✅ Heatmaps saved")

In [ ]:
# Cross-dataset comparison (if both exist)
if df_skab is not None and df_batadal is not None:
    # Average F1 scores by model across all scenarios
    skab_avg = df_skab[df_skab['metric'] == 'f1'].groupby('model')['mean'].mean()
    batadal_avg = df_batadal[df_batadal['metric'] == 'f1'].groupby('model')['mean'].mean()
    
    comparison_df = pd.DataFrame({
        'SKAB': skab_avg,
        'BATADAL': batadal_avg
    })
    
    print("\n📊 Cross-Dataset F1 Score Comparison (Average):")
    print(comparison_df.to_string())
    
    fig, ax = plt.subplots(figsize=(10, 6))
    comparison_df.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('Cross-Dataset F1 Score Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average F1 Score')
    ax.set_xlabel('Model')
    ax.set_ylim([0, 1])
    ax.legend(title='Dataset')
    ax.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(Path('../results/cross_dataset_f1_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Cross-dataset comparison saved")

In [ ]:
# Display benchmark reports
print("\n" + "="*70)
print("SKAB BENCHMARK REPORT")
print("="*70)

skab_report = RESULTS_SKAB / 'skab_benchmark_report.md'
if skab_report.exists():
    with open(skab_report, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("Report not found yet")

print("\n" + "="*70)
print("BATADAL BENCHMARK REPORT")
print("="*70)

batadal_report = RESULTS_BATADAL / 'batadal_benchmark_report.md'
if batadal_report.exists():
    with open(batadal_report, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("Report not found yet")